In [3]:
from ecdsa import SigningKey, SECP256k1
from Crypto.Hash import RIPEMD160
import hashlib
import base58

# Your priv_to_p2pkh_compressed function
def priv_to_p2pkh_compressed(priv_key):
    sk = SigningKey.from_secret_exponent(priv_key, curve=SECP256k1)
    vk = sk.get_verifying_key()
    x = vk.pubkey.point.x()
    prefix = b'\x02' if vk.pubkey.point.y() % 2 == 0 else b'\x03'
    pub_key = prefix + x.to_bytes(32, 'big')
    sha = hashlib.sha256(pub_key).digest()
    rip = RIPEMD160.new(sha).digest()
    extended = b'\x00' + rip
    checksum = hashlib.sha256(hashlib.sha256(extended).digest()).digest()[:4]
    return base58.b58encode(extended + checksum).decode()

# Your bits_off function
def bits_off(addr, targets):
    addr_bytes = base58.b58decode(addr)[1:-4]
    min_diff = float('inf')
    for target in targets:
        target_bytes = base58.b58decode(target)[1:-4]
        diff = int.from_bytes(addr_bytes, 'big') ^ int.from_bytes(target_bytes, 'big')
        bits = bin(diff).count('1')
        min_diff = min(min_diff, bits)
    return min_diff

# Test setup
priv_key = 0x15d89f02b056716b62c2c3eb793969bfca83c87a4f30f40159e2c061e8fbf
TARGET_ADDRESSES = {"1Lpu1LJAhYTWLveYW5mwi9T2kF3NKWQcJP"}  # Just this address
addr = priv_to_p2pkh_compressed(priv_key)
print(f"Generated address: {addr}")
print(f"In targets? {addr in TARGET_ADDRESSES}")
print(f"Bits off: {bits_off(addr, TARGET_ADDRESSES)}")

Generated address: 1Jr1YfgzSZ2qy586qprdND46Xh5qQE2DRA
In targets? False
Bits off: 82


In [4]:
from ecdsa import SigningKey, SECP256k1
from Crypto.Hash import RIPEMD160
import hashlib
import base58

def priv_to_p2pkh_compressed(priv_key):
    sk = SigningKey.from_secret_exponent(priv_key, curve=SECP256k1)
    vk = sk.get_verifying_key()
    x = vk.pubkey.point.x()
    prefix = b'\x02' if vk.pubkey.point.y() % 2 == 0 else b'\x03'
    pub_key = prefix + x.to_bytes(32, 'big')
    sha = hashlib.sha256(pub_key).digest()
    rip = RIPEMD160.new(sha).digest()
    extended = b'\x00' + rip
    checksum = hashlib.sha256(hashlib.sha256(extended).digest()).digest()[:4]
    return base58.b58encode(extended + checksum).decode()

priv_key = 0x15d89f02b056716b62c2c3eb793969bfca83c87a4f30f40159e2c061e8fbf
addr_comp = priv_to_p2pkh_compressed(priv_key)
print(f"Compressed: {addr_comp}")

Compressed: 1Jr1YfgzSZ2qy586qprdND46Xh5qQE2DRA


In [5]:
def priv_to_p2pkh_uncompressed(priv_key):
    sk = SigningKey.from_secret_exponent(priv_key, curve=SECP256k1)
    vk = sk.get_verifying_key()
    x = vk.pubkey.point.x()
    y = vk.pubkey.point.y()
    pub_key = b'\x04' + x.to_bytes(32, 'big') + y.to_bytes(32, 'big')
    sha = hashlib.sha256(pub_key).digest()
    rip = RIPEMD160.new(sha).digest()
    extended = b'\x00' + rip
    checksum = hashlib.sha256(hashlib.sha256(extended).digest()).digest()[:4]
    return base58.b58encode(extended + checksum).decode()

addr_uncomp = priv_to_p2pkh_uncompressed(priv_key)
print(f"Uncompressed: {addr_uncomp}")

Uncompressed: 1LfqUJqZ4TfyXdWrJTvjgXQXdckRRjJ1wU


In [6]:
!pip install pycryptodome ecdsa base58

from google.colab import files
import ecdsa
from Crypto.Hash import RIPEMD160
import hashlib
import base58

# Upload your address file
print("Please upload your address file:")
uploaded = files.upload()
uploaded_filename = list(uploaded.keys())[0]
print(f"Uploaded file: {uploaded_filename}")

# Load the addresses into a set
with open(uploaded_filename, 'r') as f:
    TARGET_ADDRESSES = set(line.strip() for line in f if line.strip())
print(f"Loaded {len(TARGET_ADDRESSES)} addresses into memory.")

# Address generation functions
def priv_to_p2pkh_compressed(priv_key):
    sk = ecdsa.SigningKey.from_secret_exponent(priv_key, curve=ecdsa.SECP256k1)
    vk = sk.get_verifying_key()
    x = vk.pubkey.point.x()
    prefix = b'\x02' if vk.pubkey.point.y() % 2 == 0 else b'\x03'
    pub_key = prefix + x.to_bytes(32, 'big')
    sha = hashlib.sha256(pub_key).digest()
    rip = RIPEMD160.new(sha).digest()
    extended = b'\x00' + rip
    checksum = hashlib.sha256(hashlib.sha256(extended).digest()).digest()[:4]
    return base58.b58encode(extended + checksum).decode()

def priv_to_p2pkh_uncompressed(priv_key):
    sk = ecdsa.SigningKey.from_secret_exponent(priv_key, curve=ecdsa.SECP256k1)
    vk = sk.get_verifying_key()
    x = vk.pubkey.point.x()
    y = vk.pubkey.point.y()
    pub_key = b'\x04' + x.to_bytes(32, 'big') + y.to_bytes(32, 'big')
    sha = hashlib.sha256(pub_key).digest()
    rip = RIPEMD160.new(sha).digest()
    extended = b'\x00' + rip
    checksum = hashlib.sha256(hashlib.sha256(extended).digest()).digest()[:4]
    return base58.b58encode(extended + checksum).decode()

# Bits off calculation
def bits_off(addr, targets):
    addr_bytes = base58.b58decode(addr)[1:-4]
    min_diff = float('inf')
    closest_addr = None
    for target in targets:
        target_bytes = base58.b58decode(target)[1:-4]
        diff = int.from_bytes(addr_bytes, 'big') ^ int.from_bytes(target_bytes, 'big')
        bits = bin(diff).count('1')
        if bits < min_diff:
            min_diff = bits
            closest_addr = target
    return min_diff, closest_addr

# Test the key from Worker 1
priv_key = 0x15d89f02b056716b62c2c3eb793969bfca83c87a4f30f40159e2c061e8fbf
addr_comp = priv_to_p2pkh_compressed(priv_key)
addr_uncomp = priv_to_p2pkh_uncompressed(priv_key)

print(f"\nTesting key: {hex(priv_key)}")
print(f"Compressed address: {addr_comp}")
print(f"Uncompressed address: {addr_uncomp}")

# Check matches
comp_match = addr_comp in TARGET_ADDRESSES
uncomp_match = addr_uncomp in TARGET_ADDRESSES
print(f"Compressed in list? {comp_match}")
print(f"Uncompressed in list? {uncomp_match}")

# If no exact match, calculate bits off
if not comp_match:
    bits_comp, closest_comp = bits_off(addr_comp, TARGET_ADDRESSES)
    print(f"Compressed: {bits_comp} bits off from {closest_comp}")
if not uncomp_match:
    bits_uncomp, closest_uncomp = bits_off(addr_uncomp, TARGET_ADDRESSES)
    print(f"Uncompressed: {bits_uncomp} bits off from {closest_uncomp}")

Please upload your address file:


Saving btc_addresses.txt to btc_addresses (1).txt
Uploaded file: btc_addresses (1).txt
Loaded 998 addresses into memory.

Testing key: 0x15d89f02b056716b62c2c3eb793969bfca83c87a4f30f40159e2c061e8fbf
Compressed address: 1Jr1YfgzSZ2qy586qprdND46Xh5qQE2DRA
Uncompressed address: 1LfqUJqZ4TfyXdWrJTvjgXQXdckRRjJ1wU
Compressed in list? False
Uncompressed in list? False
Compressed: 60 bits off from 1FhsNEonqwkubkQrkQyd8xEKLJB94P4paV
Uncompressed: 62 bits off from 1Hbg1bL6pwECNYDcnwNDp2mcx2c5V2HZA3


In [7]:
from ecdsa import SigningKey, SECP256k1
from Crypto.Hash import RIPEMD160
import hashlib
import base58

def priv_to_p2pkh_compressed(priv_key):
    sk = SigningKey.from_secret_exponent(priv_key, curve=SECP256k1)
    vk = sk.get_verifying_key()
    x = vk.pubkey.point.x()
    prefix = b'\x02' if vk.pubkey.point.y() % 2 == 0 else b'\x03'
    pub_key = prefix + x.to_bytes(32, 'big')
    sha = hashlib.sha256(pub_key).digest()
    rip = RIPEMD160.new(sha).digest()
    extended = b'\x00' + rip
    checksum = hashlib.sha256(hashlib.sha256(extended).digest()).digest()[:4]
    return base58.b58encode(extended + checksum).decode()

def priv_to_p2pkh_uncompressed(priv_key):
    sk = SigningKey.from_secret_exponent(priv_key, curve=SECP256k1)
    vk = sk.get_verifying_key()
    x = vk.pubkey.point.x()
    y = vk.pubkey.point.y()
    pub_key = b'\x04' + x.to_bytes(32, 'big') + y.to_bytes(32, 'big')
    sha = hashlib.sha256(pub_key).digest()
    rip = RIPEMD160.new(sha).digest()
    extended = b'\x00' + rip
    checksum = hashlib.sha256(hashlib.sha256(extended).digest()).digest()[:4]
    return base58.b58encode(extended + checksum).decode()

priv_key = 0x15d89f02b056716b62c2c3eb793969bfca83c87a4f30f40159e2c061e8fbf
print("Compressed:", priv_to_p2pkh_compressed(priv_key))
print("Uncompressed:", priv_to_p2pkh_uncompressed(priv_key))

Compressed: 1Jr1YfgzSZ2qy586qprdND46Xh5qQE2DRA
Uncompressed: 1LfqUJqZ4TfyXdWrJTvjgXQXdckRRjJ1wU


In [8]:
!pip install pycryptodome ecdsa base58

from google.colab import files
import ecdsa
from Crypto.Hash import RIPEMD160
import hashlib
import base58
import re

# Upload your target addresses file
print("Please upload your target addresses file:")
uploaded = files.upload()
target_filename = list(uploaded.keys())[0]
print(f"Uploaded target file: {target_filename}")

# Load target addresses and their Hash160s
TARGET_ADDRESSES = {}
with open(target_filename, 'r') as f:
    for line in f:
        addr = line.strip()
        if addr:
            hash160 = base58.b58decode(addr)[1:-4]  # Strip version byte and checksum
            TARGET_ADDRESSES[addr] = hash160
print(f"Loaded {len(TARGET_ADDRESSES)} target addresses into memory.")

# Upload your keys file (worker log format)
print("Please upload your keys file (worker log format):")
uploaded = files.upload()
keys_filename = list(uploaded.keys())[0]
print(f"Uploaded keys file: {keys_filename}")

# Address generation functions
def priv_to_p2pkh_compressed(priv_key):
    sk = ecdsa.SigningKey.from_secret_exponent(priv_key, curve=ecdsa.SECP256k1)
    vk = sk.get_verifying_key()
    x = vk.pubkey.point.x()
    prefix = b'\x02' if vk.pubkey.point.y() % 2 == 0 else b'\x03'
    pub_key = prefix + x.to_bytes(32, 'big')
    sha = hashlib.sha256(pub_key).digest()
    rip = RIPEMD160.new(sha).digest()
    extended = b'\x00' + rip
    checksum = hashlib.sha256(hashlib.sha256(extended).digest()).digest()[:4]
    return base58.b58encode(extended + checksum).decode(), rip

def priv_to_p2pkh_uncompressed(priv_key):
    sk = ecdsa.SigningKey.from_secret_exponent(priv_key, curve=ecdsa.SECP256k1)
    vk = sk.get_verifying_key()
    x = vk.pubkey.point.x()
    y = vk.pubkey.point.y()
    pub_key = b'\x04' + x.to_bytes(32, 'big') + y.to_bytes(32, 'big')
    sha = hashlib.sha256(pub_key).digest()
    rip = RIPEMD160.new(sha).digest()
    extended = b'\x00' + rip
    checksum = hashlib.sha256(hashlib.sha256(extended).digest()).digest()[:4]
    return base58.b58encode(extended + checksum).decode(), rip

# Bits off calculation
def bits_off(hash160, targets):
    min_diff = float('inf')
    closest_addr = None
    for addr, target_hash160 in targets.items():
        diff = int.from_bytes(hash160, 'big') ^ int.from_bytes(target_hash160, 'big')
        bits = bin(diff).count('1')
        if bits < min_diff:
            min_diff = bits
            closest_addr = addr
    return min_diff, closest_addr

# Parse and process keys from worker log
print("\nProcessing keys from worker log...")
key_pattern = re.compile(r"current key: (0x[0-9a-f]+)")
with open(keys_filename, 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        match = key_pattern.search(line)
        if not match:
            print(f"Skipping line (no key found): {line}")
            continue

        priv_key_hex = match.group(1)
        try:
            priv_key = int(priv_key_hex, 16)
        except ValueError:
            print(f"Skipping invalid key: {priv_key_hex}")
            continue

        # Generate both addresses
        addr_comp, hash160_comp = priv_to_p2pkh_compressed(priv_key)
        addr_uncomp, hash160_uncomp = priv_to_p2pkh_uncompressed(priv_key)

        print(f"\nLine: {line}")
        print(f"Key: {priv_key_hex}")
        print(f"Compressed: {addr_comp}")
        print(f"Uncompressed: {addr_uncomp}")

        # Check matches
        comp_match = hash160_comp in TARGET_ADDRESSES.values()
        uncomp_match = hash160_uncomp in TARGET_ADDRESSES.values()

        if comp_match:
            matched_addr = next(addr for addr, h in TARGET_ADDRESSES.items() if h == hash160_comp)
            print(f"Compressed MATCHES: {matched_addr}")
        else:
            bits_comp, closest_comp = bits_off(hash160_comp, TARGET_ADDRESSES)
            print(f"Compressed: {bits_comp} bits off from {closest_comp}")

        if uncomp_match:
            matched_addr = next(addr for addr, h in TARGET_ADDRESSES.items() if h == hash160_uncomp)
            print(f"Uncompressed MATCHES: {matched_addr}")
        else:
            bits_uncomp, closest_uncomp = bits_off(hash160_uncomp, TARGET_ADDRESSES)
            print(f"Uncompressed: {bits_uncomp} bits off from {closest_uncomp}")

Please upload your target addresses file:


Saving btc_addresses.txt to btc_addresses (2).txt
Uploaded target file: btc_addresses (2).txt
Loaded 998 target addresses into memory.
Please upload your keys file (worker log format):


Saving workerswork.txt to workerswork.txt
Uploaded keys file: workerswork.txt

Processing keys from worker log...

Line: Worker 1: 9172000 keys tested, best: 40 bits off, current key: 0x15d89f02b056716b62c2c3eb793969bfca83c87a4f30f40159e2c061e8fbf
Key: 0x15d89f02b056716b62c2c3eb793969bfca83c87a4f30f40159e2c061e8fbf
Compressed: 1Jr1YfgzSZ2qy586qprdND46Xh5qQE2DRA
Uncompressed: 1LfqUJqZ4TfyXdWrJTvjgXQXdckRRjJ1wU
Compressed: 60 bits off from 1FhsNEonqwkubkQrkQyd8xEKLJB94P4paV
Uncompressed: 62 bits off from 1Hbg1bL6pwECNYDcnwNDp2mcx2c5V2HZA3

Line: Worker 7: 9146000 keys tested, best: 39 bits off, current key: 0x1625d6c477354eec647308ffd2cd961c8777ac0eacb9a8728d549845df477
Key: 0x1625d6c477354eec647308ffd2cd961c8777ac0eacb9a8728d549845df477
Compressed: 1EqkxnZLN8JGg5NPjUrHhPWN63SsmBUqa3
Uncompressed: 1DhW5P2cPsNXsHmTST78p39nj8Ysqa7RH
Compressed: 58 bits off from 19NBWfZniu18DbmneRcaGU3sZCrCvrZrRR
Uncompressed: 58 bits off from 1GWvNXZtdFkMngrmzZ762RLy3KqDSne7He

Line: Worker 5: 9225000 keys

In [9]:
import ecdsa
import Crypto
import base58
print(f"ecdsa: {ecdsa.__version__}")
print(f"pycryptodome: {Crypto.__version__}")
print(f"base58: {base58.__version__}")

ecdsa: 0.19.1
pycryptodome: 3.22.0
base58: 2.1.1


In [10]:
from ecdsa import SigningKey, SECP256k1
from Crypto.Hash import RIPEMD160
import hashlib
import base58

priv_key = 0x15d89f02b056716b62c2c3eb793969bfca83c87a4f30f40159e2c061e8fbf
sk = SigningKey.from_secret_exponent(priv_key, curve=SECP256k1)
vk = sk.get_verifying_key()
x = vk.pubkey.point.x()
y = vk.pubkey.point.y()
prefix = b'\x02' if y % 2 == 0 else b'\x03'
pub_key = prefix + x.to_bytes(32, 'big')
sha = hashlib.sha256(pub_key).digest()
rip = RIPEMD160.new(sha).digest()
extended = b'\x00' + rip
checksum = hashlib.sha256(hashlib.sha256(extended).digest()).digest()[:4]
print("Compressed:", base58.b58encode(extended + checksum).decode())

pub_key = b'\x04' + x.to_bytes(32, 'big') + y.to_bytes(32, 'big')
sha = hashlib.sha256(pub_key).digest()
rip = RIPEMD160.new(sha).digest()
extended = b'\x00' + rip
checksum = hashlib.sha256(hashlib.sha256(extended).digest()).digest()[:4]
print("Uncompressed:", base58.b58encode(extended + checksum).decode())

Compressed: 1Jr1YfgzSZ2qy586qprdND46Xh5qQE2DRA
Uncompressed: 1LfqUJqZ4TfyXdWrJTvjgXQXdckRRjJ1wU


In [11]:
from google.colab import files
import hashlib
import base58
import re

# Upload target addresses
print("Please upload your target addresses file:")
uploaded = files.upload()
target_filename = list(uploaded.keys())[0]
print(f"Uploaded target file: {target_filename}")

TARGET_ADDRESSES = {}
with open(target_filename, 'r') as f:
    for line in f:
        addr = line.strip()
        if addr:
            hash160 = base58.b58decode(addr)[1:-4]
            TARGET_ADDRESSES[addr] = hash160
print(f"Loaded {len(TARGET_ADDRESSES)} target addresses.")

# Upload keys file (worker log)
print("Please upload your keys file (worker log):")
uploaded = files.upload()
keys_filename = list(uploaded.keys())[0]
print(f"Uploaded keys file: {keys_filename}")

# Hardcoded SECP256k1
p = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEFFFFFC2F
n = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141
Gx = 0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798
Gy = 0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8

def mod_inverse(a, m):
    def extended_gcd(a, b):
        if a == 0:
            return b, 0, 1
        gcd, x1, y1 = extended_gcd(b % a, a)
        x = y1 - (b // a) * x1
        y = x1
        return gcd, x, y
    _, x, _ = extended_gcd(a, m)
    return (x % m + m) % m

def point_add(P, Q):
    if P is None:
        return Q
    if Q is None:
        return P
    x1, y1 = P
    x2, y2 = Q
    if x1 == x2 and y1 != y2:
        return None

    if P == Q:
        lam = (3 * x1 * x1 * mod_inverse(2 * y1, p)) % p
    else:
        lam = ((y2 - y1) * mod_inverse(x2 - x1, p)) % p

    x3 = (lam * lam - x1 - x2) % p
    y3 = (lam * (x1 - x3) - y1) % p
    return (x3, y3)

def scalar_mult(k, point):
    result = None
    current = point
    while k:
        if k & 1:
            result = point_add(result, current)
        current = point_add(current, current)
        k >>= 1
    return result

def priv_to_pubkey(priv_key):
    G = (Gx, Gy)
    return scalar_mult(priv_key, G)

def pubkey_to_address(pubkey, compressed=True):
    x, y = pubkey
    if compressed:
        prefix = b'\x02' if y % 2 == 0 else b'\x03'
        pubkey_bytes = prefix + x.to_bytes(32, 'big')
    else:
        pubkey_bytes = b'\x04' + x.to_bytes(32, 'big') + y.to_bytes(32, 'big')

    sha = hashlib.sha256(pubkey_bytes).digest()
    rip = hashlib.new('ripemd160', sha).digest()
    extended = b'\x00' + rip
    checksum = hashlib.sha256(hashlib.sha256(extended).digest()).digest()[:4]
    return base58.b58encode(extended + checksum).decode(), rip

def bits_off(hash160, targets):
    min_diff = float('inf')
    closest_addr = None
    for addr, target_hash160 in targets.items():
        diff = int.from_bytes(hash160, 'big') ^ int.from_bytes(target_hash160, 'big')
        bits = bin(diff).count('1')
        if bits < min_diff:
            min_diff = bits
            closest_addr = addr
    return min_diff, closest_addr

# Process worker log
print("\nProcessing keys...")
key_pattern = re.compile(r"current key: (0x[0-9a-f]+)")
with open(keys_filename, 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        match = key_pattern.search(line)
        if not match:
            continue

        priv_key = int(match.group(1), 16)
        pubkey = priv_to_pubkey(priv_key)
        addr_comp, hash160_comp = pubkey_to_address(pubkey, compressed=True)
        addr_uncomp, hash160_uncomp = pubkey_to_address(pubkey, compressed=False)

        print(f"\nLine: {line}")
        print(f"Key: {hex(priv_key)}")
        print(f"Compressed: {addr_comp}")
        print(f"Uncompressed: {addr_uncomp}")

        comp_match = hash160_comp in TARGET_ADDRESSES.values()
        uncomp_match = hash160_uncomp in TARGET_ADDRESSES.values()

        if comp_match:
            matched_addr = next(addr for addr, h in TARGET_ADDRESSES.items() if h == hash160_comp)
            print(f"Compressed MATCHES: {matched_addr}")
        else:
            bits_comp, closest_comp = bits_off(hash160_comp, TARGET_ADDRESSES)
            print(f"Compressed: {bits_comp} bits off from {closest_comp}")

        if uncomp_match:
            matched_addr = next(addr for addr, h in TARGET_ADDRESSES.items() if h == hash160_uncomp)
            print(f"Uncompressed MATCHES: {matched_addr}")
        else:
            bits_uncomp, closest_uncomp = bits_off(hash160_uncomp, TARGET_ADDRESSES)
            print(f"Uncompressed: {bits_uncomp} bits off from {closest_uncomp}")

Please upload your target addresses file:


Saving btc_addresses.txt to btc_addresses (3).txt
Uploaded target file: btc_addresses (3).txt
Loaded 998 target addresses.
Please upload your keys file (worker log):


Saving workerswork.txt to workerswork (1).txt
Uploaded keys file: workerswork (1).txt

Processing keys...


ValueError: unsupported hash type ripemd160

In [16]:
from google.colab import files
import re
import multiprocessing

# Upload target addresses
print("Please upload your target addresses file:")
uploaded = files.upload()
target_filename = list(uploaded.keys())[0]
print(f"Uploaded target file: {target_filename}")

TARGET_ADDRESSES = {}
with open(target_filename, 'r') as f:
    for line in f:
        addr = line.strip()
        if addr:
            TARGET_ADDRESSES[addr] = None  # We’ll check addresses directly
print(f"Loaded {len(TARGET_ADDRESSES)} target addresses.")

# Upload worker log
print("Please upload your keys file (worker log):")
uploaded = files.upload()
keys_filename = list(uploaded.keys())[0]
print(f"Uploaded keys file: {keys_filename}")

# SECP256k1 parameters
P = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEFFFFFC2F
N = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141
Gx = 0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798
Gy = 0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8

def mod_inverse(a, m):
    def extended_gcd(a, b):
        if a == 0:
            return b, 0, 1
        gcd, x1, y1 = extended_gcd(b % a, a)
        x = y1 - (b // a) * x1
        y = x1
        return gcd, x, y
    _, x, _ = extended_gcd(a, m)
    return (x % m + m) % m

def point_add(P, Q):
    if P is None:
        return Q
    if Q is None:
        return P
    x1, y1 = P
    x2, y2 = Q
    if x1 == x2 and y1 != y2:
        return None

    if P == Q:
        lam = (3 * x1 * x1 * mod_inverse(2 * y1, P)) % P
    else:
        lam = ((y2 - y1) * mod_inverse(x2 - x1, P)) % P

    x3 = (lam * lam - x1 - x2) % P
    y3 = (lam * (x1 - x3) - y1) % P
    return (x3, y3)

def scalar_mult(k, point):
    result = None
    current = point
    k = k % N  # Ensure within curve order
    while k:
        if k & 1:
            result = point_add(result, current)
        current = point_add(current, current)
        k >>= 1
    return result

def priv_to_pubkey(priv_key):
    G = (Gx, Gy)
    return scalar_mult(priv_key, G)

# Hardcoded SHA256
def sha256(data):
    k = [
        0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5, 0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5,
        0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3, 0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174,
        0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc, 0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da,
        0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7, 0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967,
        0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13, 0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85,
        0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3, 0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070,
        0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5, 0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3,
        0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208, 0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2
    ]
    h = [0x6a09e667, 0xbb67ae85, 0x3c6ef372, 0xa54ff53a, 0x510e527f, 0x9b05688c, 0x1f83d9ab, 0x5be0cd19]

    data = bytearray(data)
    orig_len = len(data) * 8
    data.append(0x80)
    while (len(data) * 8) % 512 != 448:
        data.append(0)
    data += orig_len.to_bytes(8, 'big')

    for i in range(0, len(data), 64):
        w = [0] * 64
        chunk = data[i:i+64]
        for j in range(16):
            w[j] = int.from_bytes(chunk[j*4:j*4+4], 'big')
        for j in range(16, 64):
            s0 = ((w[j-15] >> 7) | (w[j-15] << 25)) ^ ((w[j-15] >> 18) | (w[j-15] << 14)) ^ (w[j-15] >> 3)
            s1 = ((w[j-2] >> 17) | (w[j-2] << 15)) ^ ((w[j-2] >> 19) | (w[j-2] << 13)) ^ (w[j-2] >> 10)
            w[j] = (w[j-16] + s0 + w[j-7] + s1) & 0xFFFFFFFF

        a, b, c, d, e, f, g, h0 = h
        for j in range(64):
            s1 = ((e >> 6) | (e << 26)) ^ ((e >> 11) | (e << 21)) ^ ((e >> 25) | (e << 7))
            ch = (e & f) ^ (~e & g)
            temp1 = (h0 + s1 + ch + k[j] + w[j]) & 0xFFFFFFFF
            s0 = ((a >> 2) | (a << 30)) ^ ((a >> 13) | (a << 19)) ^ ((a >> 22) | (a << 10))
            maj = (a & b) ^ (a & c) ^ (b & c)
            temp2 = (s0 + maj) & 0xFFFFFFFF
            h0 = g
            g = f
            f = e
            e = (d + temp1) & 0xFFFFFFFF
            d = c
            c = b
            b = a
            a = (temp1 + temp2) & 0xFFFFFFFF

        h[0] = (h[0] + a) & 0xFFFFFFFF
        h[1] = (h[1] + b) & 0xFFFFFFFF
        h[2] = (h[2] + c) & 0xFFFFFFFF
        h[3] = (h[3] + d) & 0xFFFFFFFF
        h[4] = (h[4] + e) & 0xFFFFFFFF
        h[5] = (h[5] + f) & 0xFFFFFFFF
        h[6] = (h[6] + g) & 0xFFFFFFFF
        h[7] = (h[7] + h0) & 0xFFFFFFFF

    return b''.join(x.to_bytes(4, 'big') for x in h)

# Hardcoded RIPEMD-160
def rol(x, n):
    return ((x << n) & 0xFFFFFFFF) | (x >> (32 - n))

def ripemd160(data):
    h = [0x67452301, 0xEFCDAB89, 0x98BADCFE, 0x10325476, 0xC3D2E1F0]
    r = [
        [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
        [7, 4, 13, 1, 10, 6, 15, 3, 12, 0, 9, 5, 2, 14, 11, 8],
        [3, 10, 14, 4, 9, 15, 8, 1, 2, 12, 5, 11, 6, 7, 0, 13],
        [1, 9, 11, 10, 0, 8, 12, 4, 13, 3, 7, 15, 14, 5, 6, 2],
        [4, 0, 5, 9, 7, 12, 2, 10, 14, 1, 3, 8, 11, 6, 15, 13]
    ]
    rr = [
        [5, 14, 7, 0, 9, 2, 11, 4, 13, 6, 15, 8, 1, 10, 3, 12],
        [6, 11, 3, 7, 0, 13, 5, 10, 14, 15, 8, 12, 4, 9, 1, 2],
        [15, 5, 1, 3, 7, 14, 6, 9, 11, 8, 12, 2, 10, 0, 4, 13],
        [8, 6, 4, 1, 3, 11, 15, 0, 5, 12, 2, 13, 9, 7, 10, 14],
        [12, 15, 10, 4, 1, 5, 8, 7, 6, 2, 13, 14, 0, 3, 9, 11]
    ]
    s = [
        [11, 14, 15, 12, 5, 8, 7, 9, 11, 13, 14, 15, 6, 7, 9, 8],
        [7, 6, 8, 13, 11, 9, 7, 15, 7, 12, 15, 9, 11, 7, 13, 12],
        [11, 13, 6, 7, 14, 9, 13, 15, 14, 8, 13, 6, 5, 12, 7, 5],
        [11, 13, 6, 7, 14, 9, 13, 15, 14, 8, 13, 6, 5, 12, 7, 5],
        [9, 11, 7, 13, 15, 6, 8, 14, 12, 5, 7, 14, 8, 6, 5, 12]
    ]
    k = [0x00000000, 0x5A827999, 0x6ED9EBA1, 0x8F1BBCDC, 0xA953FD4E]
    kr = [0x50A28BE6, 0x5C4DD124, 0x6D703EF3, 0x7A6D76E9, 0x00000000]

    data = bytearray(data)
    orig_len = len(data) * 8
    data.append(0x80)
    while (len(data) * 8) % 512 != 448:
        data.append(0)
    data += (orig_len & 0xFFFFFFFFFFFFFFFF).to_bytes(8, 'little')

    for i in range(0, len(data), 64):
        chunk = data[i:i+64]
        w = [int.from_bytes(chunk[j:j+4], 'little') for j in range(0, 64, 4)]
        a, b, c, d, e = h
        aa, bb, cc, dd, ee = h

        for j in range(80):
            f = (b & c) | (~b & d) if j < 16 else (b & d) | (c & ~d) if j < 32 else b ^ c ^ d if j < 48 else c ^ (b | ~d) if j < 64 else b ^ c ^ d
            ff = (bb & dd) | (cc & ~dd) if j < 16 else bb ^ cc ^ dd if j < 32 else (bb & cc) | (dd & ~bb) if j < 48 else (bb | ~cc) ^ dd if j < 64 else bb ^ cc ^ dd
            t = (a + f + k[j // 16] + w[r[j // 16][j % 16]]) % 0x100000000
            tt = (aa + ff + kr[j // 16] + w[rr[j // 16][j % 16]]) % 0x100000000
            a = e
            e = d
            d = rol(c, 10)
            c = b
            b = (rol(t, s[j // 16][j % 16]) + e) % 0x100000000
            aa = ee
            ee = dd
            dd = rol(cc, 10)
            cc = bb
            bb = (rol(tt, s[j // 16][j % 16]) + ee) % 0x100000000

        t = (h[1] + c + dd) % 0x100000000
        h[1] = (h[2] + d + ee) % 0x100000000
        h[2] = (h[3] + e + aa) % 0x100000000
        h[3] = (h[4] + a + bb) % 0x100000000
        h[4] = (h[0] + b + cc) % 0x100000000
        h[0] = t

    return b''.join(x.to_bytes(4, 'little') for x in h[:5])

# Hardcoded Base58Check
BASE58_ALPHABET = '123456789ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz'
def base58check_encode(data):
    checksum = sha256(sha256(data))[:4]
    num = int.from_bytes(data + checksum, 'big')
    result = []
    while num > 0:
        num, rem = divmod(num, 58)
        result.append(BASE58_ALPHABET[rem])
    for byte in data:
        if byte != 0:
            break
        result.append(BASE58_ALPHABET[0])
    return ''.join(result[::-1])

def pubkey_to_address(pubkey, compressed=False):
    x, y = pubkey
    if compressed:
        prefix = b'\x02' if y % 2 == 0 else b'\x03'
        pubkey_bytes = prefix + x.to_bytes(32, 'big')
    else:
        pubkey_bytes = b'\x04' + x.to_bytes(32, 'big') + y.to_bytes(32, 'big')

    sha = sha256(pubkey_bytes)
    rip = ripemd160(sha)
    return base58check_encode(b'\x00' + rip)

# Worker function
def worker(worker_id, start_key, step, target_addresses, queue, steps_per_report=1000):
    priv_key = start_key
    steps = 0
    while True:
        pubkey = priv_to_pubkey(priv_key)
        addr_uncomp = pubkey_to_address(pubkey, compressed=False)  # 2009 default
        addr_comp = pubkey_to_address(pubkey, compressed=True)

        if addr_uncomp in target_addresses:
            print(f"\n!!! WORKER {worker_id} CRACKED IT (UNCOMPRESSED) !!!\nAddress: {addr_uncomp}\nPrivate Key: {hex(priv_key)}\n")
            queue.put((priv_key, addr_uncomp))
            return
        if addr_comp in target_addresses:
            print(f"\n!!! WORKER {worker_id} CRACKED IT (COMPRESSED) !!!\nAddress: {addr_comp}\nPrivate Key: {hex(priv_key)}\n")
            queue.put((priv_key, addr_comp))
            return

        steps += 1
        if steps % steps_per_report == 0:
            print(f"Worker {worker_id}: {steps} keys tested, current key: {hex(priv_key)}")

        priv_key = (priv_key + step) % N  # Simple increment, adjust if needed

# Parse worker log and start workers
print("\nStarting workers (2009-style)...")
key_pattern = re.compile(r"Worker (\d+): \d+ keys tested, best: \d+ bits off, current key: (0x[0-9a-f]+)")
worker_keys = {}
with open(keys_filename, 'r') as f:
    for line in f:
        match = key_pattern.search(line.strip())
        if match:
            worker_id, priv_key = int(match.group(1)), int(match.group(2), 16)
            worker_keys[worker_id] = priv_key

processes = []
queue = multiprocessing.Queue()
for worker_id, start_key in worker_keys.items():
    step = worker_id + 1  # Unique step per worker
    p = multiprocessing.Process(target=worker, args=(worker_id, start_key, step, TARGET_ADDRESSES, queue))
    processes.append(p)
    p.start()

# Wait for a crack or manual stop
for p in processes:
    p.join()

if not queue.empty():
    priv_key, addr = queue.get()
    print(f"\nFinal Crack: Address: {addr}, Private Key: {hex(priv_key)}")
else:
    print("\nNo keys cracked—workers exhausted or interrupted.")

Please upload your target addresses file:


Saving btc_addresses.txt to btc_addresses (8).txt
Uploaded target file: btc_addresses (8).txt
Loaded 998 target addresses.
Please upload your keys file (worker log):


Saving workerswork.txt to workerswork (6).txt
Uploaded keys file: workerswork (6).txt

Starting workers (2009-style)...


Process Process-13:
Traceback (most recent call last):
  File "/usr/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
Process Process-14:
  File "/usr/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "<ipython-input-16-0dab03009dbe>", line 235, in worker
    pubkey = priv_to_pubkey(priv_key)
             ^^^^^^^^^^^^^^^^^^^^^^^^
Traceback (most recent call last):
  File "<ipython-input-16-0dab03009dbe>", line 74, in priv_to_pubkey
    return scalar_mult(priv_key, G)
           ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "<ipython-input-16-0dab03009dbe>", line 68, in scalar_mult
    current = point_add(current, current)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "<ipython-input-16-0da


No keys cracked—workers exhausted or interrupted.


In [21]:
# Test with priv_key = 1 (should return G as pubkey)
priv_key = 1
pubkey = priv_to_pubkey(priv_key)
addr_uncomp = pubkey_to_address(pubkey, compressed=False)
print(f"Test: priv_key=1, uncompressed address={addr_uncomp}")

mod_inverse: a=65341020041517633956166170261014086368942546761318486551877808671514674964848, m=(55066263022277343669578718895168534326250603453777594175500187360389116729240, 32670510020758816978083085130507043184471273380659243275938904335757337482424)


TypeError: unsupported operand type(s) for %: 'int' and 'tuple'